# 21 — Zones tampons et anneaux de distance

**Objectif :** construire des bandes de distance et contrôler leurs surfaces,
leurs limites et les recouvrements. Les points, lignes et rectangles de ce cours
sont **simulés** ; ils ne représentent aucune observation de terrain.

Le moteur `buffer_rings` est commun à Python, au terminal, aux plans et à
**Vecteur → Anneaux de distance avec bilan**. Les tampons simples, dont les
tampons négatifs, restent accessibles par `buffer`.
[Contrat G12](../../docs/VECTOR_RINGS.md).

In [1]:
import hashlib
import json
import math
from pathlib import Path
import subprocess
import sys
from threading import Event
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString, box
import cartomize as cm
from cartomize.vector_io import write_geopackage
from cartomize.viewer_data import table_page
from course import run_directory

work = run_directory('21_anneaux')
assert tuple(map(int, cm.__version__.split('.')[:2])) >= (1, 12)
print('Cartomize', cm.__version__)

Cartomize 1.12.0


## 1. Une liste de bornes supérieures, en mètres

`[100, 250, 500, 1000]` produit les bandes 0–100, 100–250, 250–500 et
500–1000 m. Les valeurs doivent être strictement positives et croissantes.
Le logiciel ne les trie pas et ne supprime pas les doublons.

Le mode `dissolved`, choisi par défaut, réunit les tampons de toutes les sources
**à chaque distance**, puis retire les bandes précédentes. La distance concerne
donc la source la plus proche. Les bandes partagent éventuellement une frontière,
sans partager leur intérieur au-delà des résidus numériques contrôlés.

EPSG:32733 est ici le repère métrique d'un exemple simulé. Une étude réelle doit
choisir une projection adaptée à son emprise ; aucun tampon en degrés ni calcul
géodésique implicite n'est effectué.

In [2]:
points = gpd.GeoDataFrame({'code': ['001', 'NA']}, geometry=[
    Point(300000, 9500000), Point(300450, 9500000)], crs=32733)
source_wkb = points.geometry.to_wkb().tolist()
bounds = [100, 250, 500, 1000]
merged = cm.buffer_rings(points, bounds, id_field='code', area_unit='m2')
assert merged.table['lower_m'].tolist() == [0., 100., 250., 500.]
assert merged.table['upper_m'].tolist() == [100., 250., 500., 1000.]
assert merged.summary['group_count'] == 1
assert merged.sources['id_value'].tolist() == ['001', 'NA']
assert merged.balances['overlap_area'].max() < 1e-6
display(merged.table)
display(merged.balances)

,ring_id,group_id,band_index,lower_m,upper_m,area
0,r000001,g000001,1,0.0,100.0,6.273097e+04
1,r000002,g000001,2,100.0,250.0,3.220879e+05
2,r000003,g000001,3,250.0,500.0,8.332500e+05
3,r000004,g000001,4,500.0,1000.0,2.810123e+06


,group_id,source_count,base_area,outer_area,expected_area,sum_ring_area,union_ring_area,overlap_area,excess_area,raw_excess_area,uncovered_area,outside_area,closure_error,max_step_closure_error,non_nested_area
0,g000001,2,0.0,4.028192e+06,4.028192e+06,4.028192e+06,4.028192e+06,0.0,0.0,-2.328306e-09,0.0,0.0,9.313226e-10,9.313226e-10,0.0


## 2. Vérification indépendante sur un point

Les courbes sont polygonales : `quad_segs=q` place `q` segments par quart de
cercle. L'aire du polygone régulier inscrit autour d'un point est

$$A_q(r)=2q\,\sin\left(\frac{\pi}{2q}\right)r^2.$$

L'aire attendue d'une couronne est `A_q(r_ext) − A_q(r_int)`. Cette formule
arithmétique ne fait appel ni à un tampon ni à une intersection géométrique.
Elle distingue l'approximation du cercle et les résidus du calcul de surface.

In [3]:
q = 16
factor = 2 * q * math.sin(math.pi / (2 * q))
single = cm.buffer_rings(points.iloc[:1], bounds, quad_segs=q, area_unit='m2')
expected = np.diff([0.] + [factor * radius**2 for radius in bounds])
np.testing.assert_allclose(single.table['area'], expected, rtol=1e-10, atol=1e-6)
print('Écart relatif entre le disque polygonal et le disque idéal :',
      f'{100 * (1 - factor / math.pi):.6f} %')
display(single.table.assign(expected_m2=expected))

Écart relatif entre le disque polygonal et le disque idéal : 0.160561 %


,ring_id,group_id,band_index,lower_m,upper_m,area,expected_m2
0,r000001,g000001,1,0.0,100.0,3.136548e+04,3.136548e+04
1,r000002,g000001,2,100.0,250.0,1.646688e+05,1.646688e+05
2,r000003,g000001,3,250.0,500.0,5.881028e+05,5.881028e+05
3,r000004,g000001,4,500.0,1000.0,2.352411e+06,2.352411e+06


## 3. Chaque source séparément : ne pas confondre somme et surface unique

Le mode `independent` produit une série d'anneaux par ligne source. Des étiquettes
identiques ne fusionnent pas les objets. Les anneaux de sources différentes
peuvent se recouvrir. `union_ring_area` mesure leur surface unique ;
`sum_ring_area` additionne les lignes ; `excess_area` est la différence.
`overlap_area` mesure une seule fois le territoire compté au moins deux fois,
y compris lorsqu'il est couvert par trois objets ou davantage.

In [4]:
independent = cm.buffer_rings(points, bounds, mode='independent',
                              id_field='code', area_unit='m2')
assert len(independent.table) == len(points) * len(bounds)
totals = independent.summary['totals']
assert totals['overlap_area'] > 0 and totals['excess_area'] > 0
assert totals['sum_ring_area'] > totals['union_ring_area']
np.testing.assert_allclose(totals['union_ring_area'],
                          merged.summary['totals']['union_ring_area'], rtol=1e-10, atol=1e-6)
display(independent.sources.drop(columns='geometry'))
display(independent.balances)
print(json.dumps(totals, indent=2, ensure_ascii=False))

,source_id,source_row,id_type,id_value,geometry_type,source_area
0,s000001,0,string,001,Point,0.0
1,s000002,1,string,NA,Point,0.0


,group_id,source_count,base_area,outer_area,expected_area,sum_ring_area,union_ring_area,overlap_area,excess_area,raw_excess_area,uncovered_area,outside_area,closure_error,max_step_closure_error,non_nested_area
0,g000001,1,0.0,3.136548e+06,3.136548e+06,3.136548e+06,3.136548e+06,0.0,0.0,-9.313226e-10,0.0,0.0,4.656613e-10,9.313226e-10,0.0
1,g000002,1,0.0,3.136548e+06,3.136548e+06,3.136548e+06,3.136548e+06,0.0,0.0,-9.313226e-10,0.0,0.0,4.656613e-10,9.313226e-10,0.0


{
  "sum_ring_area": 6273096.981090327,
  "union_ring_area": 4028192.2061842163,
  "overlap_area": 2244904.774906012,
  "excess_area": 2244904.7749061105,
  "raw_excess_area": 2244904.7749061105
}


## 4. Anneaux autour d'un axe linéaire

Pour un segment droit de longueur `L`, le tampon arrondi a une aire
`2 × L × r + A_q(r)`. La différence de deux aires cumulées donne chaque bande.
Les distances suivantes illustrent un découpage de proximité à un axe :
0–50, 50–100, 100–250, 250–500 et 500–1000 m.

In [5]:
line = gpd.GeoDataFrame({'name': ['Axe simulé']}, geometry=[
    LineString([(300000, 9500000), (302000, 9500000)])], crs=32733)
line_bounds = [50, 100, 250, 500, 1000]
line_rings = cm.buffer_rings(line, line_bounds, id_field='name', area_unit='m2')
cumulative = [0.] + [2 * 2000 * r + factor * r**2 for r in line_bounds]
np.testing.assert_allclose(line_rings.table['area'], np.diff(cumulative),
                          rtol=1e-10, atol=1e-6)
display(line_rings.table)

,ring_id,group_id,band_index,lower_m,upper_m,area
0,r000001,g000001,1,0.0,50.0,2.078414e+05
1,r000002,g000001,2,50.0,100.0,2.235241e+05
2,r000003,g000001,3,100.0,250.0,7.646688e+05
3,r000004,g000001,4,250.0,500.0,1.588103e+06
4,r000005,g000001,5,500.0,1000.0,4.352411e+06


## 5. Intérieur des polygones et couronnes extérieures

Par défaut, l'intérieur du polygone source est retiré. `include_input=True`
l'inclut dans la première bande. Les trous d'un polygone font partie de son
extérieur : la construction peut aussi créer des couronnes dans ces trous.
Ce paramètre est sans effet sur l'aire propre d'un point ou d'une ligne.

In [6]:
polygon = gpd.GeoDataFrame({'name': ['Rectangle simulé']},
    geometry=[box(300000, 9500000, 300200, 9500100)], crs=32733)
outside = cm.buffer_rings(polygon, [50, 100], area_unit='m2')
included = cm.buffer_rings(polygon, [50, 100], include_input=True, area_unit='m2')
expected_outside = np.diff([0., 600 * 50 + factor * 50**2,
                          600 * 100 + factor * 100**2])
np.testing.assert_allclose(outside.table['area'], expected_outside, rtol=1e-10, atol=1e-6)
np.testing.assert_allclose(included.table['area'] - outside.table['area'],
                          [20000., 0.], rtol=1e-10, atol=1e-6)
display(outside.balances)
display(included.balances)

,group_id,source_count,base_area,outer_area,expected_area,sum_ring_area,union_ring_area,overlap_area,excess_area,raw_excess_area,uncovered_area,outside_area,closure_error,max_step_closure_error,non_nested_area
0,g000001,1,20000.0,111365.484905,91365.484905,91365.484905,91365.484905,0.0,0.0,-2.910383e-11,0.0,0.0,-1.455192e-11,7.275958e-11,0.0


,group_id,source_count,base_area,outer_area,expected_area,sum_ring_area,union_ring_area,overlap_area,excess_area,raw_excess_area,uncovered_area,outside_area,closure_error,max_step_closure_error,non_nested_area
0,g000001,1,0.0,111365.484905,111365.484905,111365.484905,111365.484905,0.0,0.0,-4.365575e-11,0.0,0.0,0.0,7.275958e-11,0.0


## 6. Contrôle visuel des deux modes et des axes

La carte aide à interpréter les résultats ; les assertions précédentes assurent
les contrôles numériques. Tous les objets représentés restent des simulations.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
merged.data.plot(column='upper_m', categorical=True, cmap='viridis',
                 edgecolor='white', linewidth=.7, legend=True, ax=axes[0])
points.plot(color='#b53737', markersize=25, ax=axes[0])
axes[0].set_title('Points simulés : anneaux fusionnés')
line_rings.data.plot(column='upper_m', categorical=True, cmap='viridis',
                    edgecolor='white', linewidth=.7, legend=True, ax=axes[1])
line.plot(color='#b53737', linewidth=2, ax=axes[1])
axes[1].set_title('Axe simulé : bandes de distance')
for ax in axes:
    ax.set_aspect('equal')
    ax.set_xlabel('Est — SCR projeté, mètres')
    ax.set_ylabel('Nord — SCR projeté, mètres')
plt.show()

## 7. Publication complète et relecture

`execute_operation` écrit un nouveau dossier : anneaux et registre des sources
en GeoPackage, intervalles et bilans en CSV avec schéma, audit, résumé et
manifeste. Chaque géométrie, attribut et valeur de table est relu avant la
publication atomique. Un échec ou une annulation ne laisse pas de résultat
partiel. Le registre conserve les étiquettes comme `001` et `NA` exactement.

In [8]:
source_path = work / 'sources.gpkg'
write_geopackage(points, source_path, layer='007')
source_sha = hashlib.sha256(source_path.read_bytes()).hexdigest()
parameters = dict(data=str(source_path), distances=bounds, layer='007',
                  id_field='code', mode='dissolved', area_unit='m2')
published = cm.execute_operation('vector.buffer_rings', parameters, work / 'published')
assert len(list((work / 'published').iterdir())) == 9
registry = cm.read_file(published['products']['sources'])
assert registry['id_value'].tolist() == ['001', 'NA']
audit = json.loads(Path(published['products']['audit']).read_text(encoding='utf-8'))
assert audit['schema'] == 'cartomize.vector_rings.audit.v1'
for filename, proof in audit['verified_files'].items():
    assert hashlib.sha256((work / 'published' / filename).read_bytes()).hexdigest() == proof['sha256']
display(table_page(published['products']['balances'])['frame'])

,group_id,source_count,base_area,outer_area,expected_area,sum_ring_area,union_ring_area,overlap_area,excess_area,raw_excess_area,uncovered_area,outside_area,closure_error,max_step_closure_error,non_nested_area
0,g000001,2,0.0,4028192.206184,4028192.206184,4028192.206184,4028192.206184,0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0


## 8. La même opération au terminal

Une commande dédiée expose les bornes, le mode, le SCR, les unités et les autres
choix. Le processus ci-dessous active explicitement UTF-8, y compris sous
Windows, pour relire correctement les accents de ses résultats.

In [9]:
command = [sys.executable, '-X', 'utf8', '-m', 'cartomize', 'buffer-rings',
           str(source_path), str(work / 'cli'), '--distances', '100', '250', '500', '1000',
           '--layer', '007', '--id-field', 'code', '--area-unit', 'm2']
completed = subprocess.run(command, check=True, capture_output=True,
                           text=True, encoding='utf-8')
cli_result = json.loads(completed.stdout)
assert cli_result['metadata']['band_count'] == 4
assert cm.read_file(cli_result['products']['sources'])['id_value'].tolist() == ['001', 'NA']
print('Commande exécutée ; quatre intervalles vérifiés.')

Commande exécutée ; quatre intervalles vérifiés.


## 9. Chaîner G12 et G09 : occupation du sol par bande de distance

Deux catégories polygonales simulées couvrent ici toute l'emprise des anneaux.
Un plan produit les bandes, puis transmet leur produit `data` au croisement des
surfaces G09. L'identifiant `ring_id` devient l'identifiant de zone. Ce parcours
prépare des tableaux de proximité ; il ne constitue pas à lui seul une analyse
statistique d'un effet de route ou d'habitat.

In [10]:
classes = gpd.GeoDataFrame({'cover': ['Végétation simulée', 'Sol simulé']},
    geometry=[box(298000, 9498000, 300200, 9502000),
              box(300200, 9498000, 303000, 9502000)], crs=32733)
classes_path = work / 'classes.gpkg'
write_geopackage(classes, classes_path)
plan = cm.processing_plan([
    dict(id='rings', operation='vector.buffer_rings', parameters=parameters),
    dict(id='areas', operation='vector.tabulate_area', parameters=dict(
        zones='@rings:data', classes=str(classes_path), zone_field='ring_id',
        class_field='cover', area_unit='m2')),
])
manifest = Path(cm.run_plan(plan, work / 'plan'))
plan_result = json.loads(manifest.read_text(encoding='utf-8'))
area_table = table_page(plan_result['products']['areas']['table'])['frame']
np.testing.assert_allclose(area_table.groupby('zone_id')['percentage'].sum(), 100., atol=1e-8)
np.testing.assert_allclose(area_table.groupby('zone_id')['area'].sum(), merged.table['area'],
                          rtol=1e-10, atol=1e-6)
display(area_table)

,zone_id,zone_type,zone_value,class_id,class_type,class_value,area,percentage
0,z000001,string,r000001,c000001,string,Végétation simulée,31365.484905,50.0
1,z000001,string,r000001,c000002,string,Sol simulé,31365.484905,50.0
2,z000002,string,r000002,c000001,string,Végétation simulée,154514.698904,47.972834
3,z000002,string,r000002,c000002,string,Sol simulé,167573.214094,52.027166
4,z000003,string,r000003,c000001,string,Végétation simulée,400557.620472,48.071722
5,z000003,string,r000003,c000002,string,Sol simulé,432692.369612,51.928278
6,z000004,string,r000004,c000001,string,Végétation simulée,1378837.079656,49.066782
7,z000004,string,r000004,c000002,string,Sol simulé,1431286.253636,50.933218


## 10. Refus explicites, annulation et intégrité des sources

Une liste désordonnée, répétée ou non positive est une erreur à corriger. Les
sources ne sont ni réparées, ni déplacées, ni écrasées. Une géométrie invalide ou
un SCR absent doit être traité explicitement avant de lancer le calcul.

In [11]:
for invalid in ([100, 50], [100, 100], [0, 100], [-50, 100]):
    try:
        cm.buffer_rings(points, invalid)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Liste invalide acceptée : {invalid}')
cancel = Event()
cancel.set()
try:
    cm.execute_operation('vector.buffer_rings', parameters, work / 'cancelled', cancel=cancel)
except cm.ProcessingCancelled:
    pass
else:
    raise AssertionError('Annulation ignorée')
assert not (work / 'cancelled').exists()
assert hashlib.sha256(source_path.read_bytes()).hexdigest() == source_sha
assert points.geometry.to_wkb().tolist() == source_wkb
print('Sources intactes ; erreurs et annulation contrôlées.')

Sources intactes ; erreurs et annulation contrôlées.


## Exercices et réponses

1. Remplacer les bornes des points par `[250, 500, 1000]`. Combien de lignes en
   mode fusionné et en mode indépendant pour deux sources ? **3 et 6**.
2. Passer de `quad_segs=16` à `32`. Quelle précision change ? **L'approximation
   polygonale des arcs, pas la précision des coordonnées de la source.**
3. Retrouver l'aire en hectares de la couronne 500–1000 m autour d'un point
   avec `q=16`. **`A_16(1000) − A_16(500)`, divisée par 10 000.**
4. Deux sources portent `001`. Sont-elles fusionnées en mode indépendant ?
   **Non : leurs positions sources et leurs identifiants techniques restent distincts.**
5. Peut-on additionner les surfaces des couronnes indépendantes pour mesurer
   une emprise unique ? **Seulement en l'absence de recouvrement ; sinon utiliser
   `union_ring_area` et examiner le bilan.**

La fenêtre de bureau donne accès aux mêmes anneaux, intervalles, sources,
bilans, résumé et audit. Les résultats ouverts restent ceux du calcul terminé,
même si l'on change ensuite les valeurs du formulaire.

In [12]:
answer_ha = factor * (1000**2 - 500**2) / 10000
exercise = cm.buffer_rings(points.iloc[:1], [500, 1000], area_unit='ha')
np.testing.assert_allclose(exercise.table.iloc[1]['area'], answer_ha, rtol=1e-10, atol=1e-8)
print(f'Couronne polygonale 500–1000 m : {answer_ha:.6f} ha (q=16).')

Couronne polygonale 500–1000 m : 235.241137 ha (q=16).
